# Comparación: Regresión Logística vs SVM vs KNN
**Dataset:** `Driver_Behavior.csv`



**Consejo:** **NO** escales antes de dividir en train/test. Primero split, luego fit del scaler solo con train.


### Objetivo
Construir y **comparar** tres modelos de clasificación (Logística, KNN y SVM) para predecir el comportamiento/riesgo del conductor.

### Bloques
- Bloque 1: Exploración
- Bloque 2: Preparación
- Bloque 3: Regresión logística
- Bloque 4: KNN
- Bloque 5: SVM
- Bloque 6: Comparación final

### Preguntas iniciales
1. ¿Por qué KNN es sensible al escalado?
2. ¿Qué efecto tiene aumentar el parámetro C en SVM?
3. ¿En qué se diferencia la frontera de decisión de Logística y SVM?
4. ¿Por qué la Accuracy puede no ser una buena métrica en problemas desbalanceados?



## Preparación
1. Coloca el archivo `Driver_Behavior.csv` en la **misma carpeta** que esté tu notebook, o ajusta la ruta.
2. Ejecuta celda a celda.
3. Mantén el notebook limpio: salidas relevantes, gráficos claros, comentarios breves.


In [44]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

import matplotlib.pyplot as plt


In [45]:
# Carga del dataset
path = "Driver_Behavior.csv"
df_drivers = pd.read_csv(path)

display(df_drivers.head())
print("Shape:", df_drivers.shape)
df_drivers.info()


,speed_kmph,accel_x,accel_y,brake_pressure,steering_angle,throttle,lane_deviation,phone_usage,headway_distance,reaction_time,behavior_label
0,36.075011,0.535763,0.708633,23.107812,-3.169956,53.123505,0.851871,1,17.996005,1.400050,Distracted
1,38.090536,0.973764,0.044312,36.961137,-24.380082,36.383904,1.459495,1,29.904182,1.428537,Distracted
2,71.314445,3.638434,0.789375,79.734087,-6.100238,78.110507,0.254723,0,11.126012,0.406950,Aggressive
3,86.485997,2.441366,0.039135,45.007002,17.886191,82.794935,0.911664,0,11.064505,0.539964,Aggressive
4,52.816777,-0.201763,0.560619,38.759612,-4.104323,61.432375,1.591244,1,21.967570,1.369908,Distracted


Shape: (30000, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   speed_kmph        30000 non-null  float64
 1   accel_x           30000 non-null  float64
 2   accel_y           30000 non-null  float64
 3   brake_pressure    30000 non-null  float64
 4   steering_angle    30000 non-null  float64
 5   throttle          30000 non-null  float64
 6   lane_deviation    30000 non-null  float64
 7   phone_usage       30000 non-null  int64  
 8   headway_distance  30000 non-null  float64
 9   reaction_time     30000 non-null  float64
 10  behavior_label    30000 non-null  object 
dtypes: float64(9), int64(1), object(1)
memory usage: 2.5+ MB



# Bloque 1 – Exploración

Completa:

1. Identifica **variable objetivo** (y) y **predictoras** (X).
2. Muestra:
   - `df.describe()` (si tiene sentido)
   - Recuento por clase de la variable objetivo
   - Valores nulos por columna
3. Responde (2–4 líneas cada una):
   - ¿Está balanceada la variable objetivo?
   - ¿Qué variables crees que pueden ser más predictoras?
   - ¿Conviene escalar? ¿Por qué?

💡 Consejo: si la variable objetivo está en texto (p.ej. "Safe"/"Aggressive"), necesitarás codificarla o usarla tal cual según scikit-learn (normalmente admite etiquetas tipo string).


In [58]:
df_drivers["behavior_label"].value_counts()

behavior_label
2    10000
0    10000
1    10000
Name: count, dtype: int64

In [56]:
df_drivers.replace({"behavior_label": {"Aggressive": 0, "Safe": 1, "Distracted": 2}}, inplace=True)
X = df_drivers.drop("behavior_label", axis=1)
y = df_drivers["behavior_label"].to_frame()

df_drivers.head()
df_drivers.corr(numeric_only=True)["behavior_label"].abs().sort_values(ascending=False)[1:]


reaction_time       0.918236
phone_usage         0.866025
speed_kmph          0.688413
accel_x             0.676367
throttle            0.568260
lane_deviation      0.486292
brake_pressure      0.458010
accel_y             0.282542
headway_distance    0.257222
steering_angle      0.000531
Name: behavior_label, dtype: float64


# Bloque 2 – Preparación

1. Divide en train/test (80/20), con `random_state=42`.
2. Aplica **StandardScaler**:
   - `fit` SOLO con X_train
   - `transform` en X_train y X_test
3. Justifica por qué el escalado es importante para KNN y SVM.


In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



## Funciones de ayuda

- accuracy
- matriz de confusión
- classification_report
- ROC y AUC (si el modelo ofrece `predict_proba` o `decision_function`)


In [57]:

# Recorre k impares (puedes consultar la teoría)
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=7)
# Calcula y guarda:
# - accuracy (en test)
# - AUC (en test) usando predict_proba
knn.fit(X_train_scaled, y_train)
y_pred = knn.predict(X_test_scaled)
y_pred_proba = knn.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
#auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')

print("Accuracy (KNN): ", accuracy)
#print("AUC (KNN): ", auc)
print("Clasification report: ", classification_report(y_test, y_pred))
confusion_matrix_result = confusion_matrix(y_test, y_pred)
print("Confusion Matrix: \n", confusion_matrix_result)



/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Accuracy (KNN):  1.0
Clasification report:                precision    recall  f1-score   support

           0       1.00      1.00      1.00      2000
           1       1.00      1.00      1.00      2000
           2       1.00      1.00      1.00      2000

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000

Confusion Matrix: 
 [[2000    0    0]
 [   0 2000    0]
 [   0    0 2000]]



# Bloque 3 – Regresión Logística

1. Entrena una Regresión Logística.
2. Evalúa con:
   - Accuracy
   - Matriz de confusión
   - Classification report
   - ROC y AUC (si procede)
3. Interpreta:
   - ¿Qué tipo de error es más preocupante en este problema y por qué?


In [60]:


# Entrena LogisticRegression (max_iter suficiente)
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(max_iter=1200)
logreg.fit(X_train_scaled, y_train)
y_pred_logreg = logreg.predict(X_test_scaled)
y_pred_proba_logreg = logreg.predict_proba(X_test_scaled)[:, 1]



# Evalúa:
# - accuracy
# - matriz de confusión
# - AUC
# - curva ROC
accuracy_logreg = accuracy_score(y_test, y_pred_logreg)
conf_matrix_logreg = confusion_matrix(y_test, y_pred_logreg)
#auc_logreg = roc_auc_score(y_test, y_pred_proba_logreg)
print("Accuracy (Logistic Regression): ", accuracy_logreg)
print("Confusion Matrix (Logistic Regression): \n", conf_matrix_logreg)
#print("AUC (Logistic Regression): ", auc_logreg)

Accuracy (Logistic Regression):  1.0
Confusion Matrix (Logistic Regression): 
 [[2000    0    0]
 [   0 2000    0]
 [   0    0 2000]]


/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



# Bloque 4 – KNN

1. Entrena KNN con `k=5`.
2. Evalúa con las mismas métricas.
3. Prueba al menos 3 valores distintos de k (por ejemplo: 1, 5, 11, 21).
4. Representa **Accuracy vs k** en un gráfico.
5. Elige el k "óptimo" y justifica tu elección.



# Bloque 5 – SVM

1. Entrena:
   - SVM lineal (`kernel="linear"`)
   - SVM RBF (`kernel="rbf"`)
2. Evalúa ambos modelos con las mismas métricas.
3. Modifica el parámetro **C** (por ejemplo: 0.1, 1, 10) y analiza el efecto.
4. Indica qué kernel funciona mejor y por qué.


In [61]:
from sklearn.svm import SVC, LinearSVC

# Entrena SVM razonadamente
# --- ENTRENAR SVM LINEAL ---
svm_linear = LinearSVC(
    C=0.01,
    max_iter=1000
)

svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)

print("🔵 SVM LINEAL - RESULTADOS:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_linear):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_linear))
print("Classification Report:\n", classification_report(y_test, y_pred_linear))

# Evalúa (accuracy + AUC + ROC)

🔵 SVM LINEAL - RESULTADOS:
Accuracy: 1.0000
Confusion Matrix:
 [[2000    0    0]
 [   0 2000    0]
 [   0    0 2000]]
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00      2000
           1       1.00      1.00      1.00      2000
           2       1.00      1.00      1.00      2000

    accuracy                           1.00      6000
   macro avg       1.00      1.00      1.00      6000
weighted avg       1.00      1.00      1.00      6000



/home/ciabd12/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)



# Bloque 6 – Comparación final

1. Crea una **tabla comparativa** con:
   - Accuracy
   - Recall (macro o de la clase positiva)
   - F1-score (macro o de la clase positiva)
   - AUC (si es binario)
2. Recomienda un modelo y justifica tu elección.

En problemas de seguridad, suele importar especialmente el **Recall** de la clase de riesgo.



# Preguntas

Responde brevemente:

1. ¿Por qué KNN es sensible al escalado? Porque a la hora de calcular las distancia, si los datos están sin escalar, tendria que usar mas potencia y las distancias son diferentes
2. ¿Qué efecto tiene aumentar el parámetro C en SVM? Hace que la recta se desvie mas o menos, por lo que si ponemos un C muy alto, podría provocar overfitting y en el caso contrario underfitting
3. ¿En qué se diferencia la frontera de decisión de Logística y SVM? la frontera de decisión mira todos los puntos posibles e intenta ajustarse a ellos para que estén lo mas lejos posible, y SVM mira los puntos mas cercanos al borde, los puntos lejanos los ignora.
4. ¿Por qué la Accuracy puede no ser una buena métrica en problemas desbalanceados? Porque puede detectar patrones que hacen que el modelo de un resultado especifico y también las predicciones saldrán desbalanceadas.


# Ayuda para las explicaciones
A la hora de responder a las preguntas, puedes ayudarte de esta pequeña guía.

1. Cada conclusión debe incluir:

    - Una métrica concreta
    - Una comparación explícita
    - Una consecuencia técnica


2. Siempre debes responder

    - ¿Qué significa esa métrica?
    - ¿Por qué importa aquí?
    - ¿Qué implicación tiene?

3. Cuando compares modelos debes responder a 3 preguntas (más o menos):
    - ¿Cuál rinde mejor globalmente? (Accuracy / AUC)
    - ¿Cuál detecta mejor la clase importante? (Recall)
    - ¿Cuál es más estable o generalizable?

>Estas notas son orientativas pero pueden ayudarte a la hora de mejorar tus respuestas.